In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EVisionary — Cross-Script Consistency Check

Purpose
-------
Guarantees that every concept count is identical regardless of which script
computes it. All scripts import a single set of canonical patterns and helpers
from evisionary_common; this check recomputes each canonical count from that
single source and asserts it against the expected reference values that were
locked in after the final pipeline run.

"""

import sys
import pandas as pd

from evisionary_common import (
    DATA_PATH_MASTER, MOLECULE_CONCEPTS, QUERY_PATTERNS,
    valid_text_series, contains_ci, count_unique_pmids,
)

# ---------------------------------------------------------
# Reference values locked after the final pipeline run.
# These are the single agreed numbers that MUST appear identically
# in the benchmark, validation, advanced_audits, and gain scripts.
# ---------------------------------------------------------
EXPECTED = {
    "dataset_rows": 258460,

    # Molecule classes (canonical molecule_type, ^class$ anchored).
    "Protein_records": 207623,
    "Protein_pmids": 569,
    "mRNA_records": 26701,
    "mRNA_pmids": 26,
    "miRNA_records": 16131,
    "miRNA_pmids": 120,
    "Lipid_records": 2896,
    "Lipid_pmids": 52,

    # Free-text / contextual concepts.
    "Plasma_records": 547,
    "Plasma_pmids": 444,
    "BreastCancer_records": 12,
    "BreastCancer_pmids": 11,

    # Species (word-boundary; includes multi-species co-occurrence).
    "HomoSapiens_records": 200644,
}

TOLERANCE = 0  # exact-match policy; counts must be identical.


def check(label, observed, expected, results):
    ok = abs(observed - expected) <= TOLERANCE
    status = "PASS" if ok else "FAIL"
    results.append((label, expected, observed, status))
    return ok


def main():
    df = pd.read_parquet(DATA_PATH_MASTER)
    results = []
    all_ok = True

    # --- Dataset size ---
    all_ok &= check("dataset_rows", len(df), EXPECTED["dataset_rows"], results)

    # --- Molecule classes (canonical molecule_type) ---
    mol = valid_text_series(df["molecule_type"])
    for concept, _exact_val, pattern in MOLECULE_CONCEPTS:
        m = contains_ci(mol, pattern)
        rec = int(m.sum())
        pm = count_unique_pmids(df, m)
        all_ok &= check(f"{concept}_records", rec,
                        EXPECTED[f"{concept}_records"], results)
        all_ok &= check(f"{concept}_pmids", pm,
                        EXPECTED[f"{concept}_pmids"], results)

    # --- Plasma (sample_name, word-boundary) ---
    m_plasma = contains_ci(valid_text_series(df["sample_name"]), QUERY_PATTERNS["Plasma"])
    all_ok &= check("Plasma_records", int(m_plasma.sum()),
                    EXPECTED["Plasma_records"], results)
    all_ok &= check("Plasma_pmids", count_unique_pmids(df, m_plasma),
                    EXPECTED["Plasma_pmids"], results)

    # --- Breast cancer (disease, word-boundary) ---
    m_bc = contains_ci(valid_text_series(df["disease"]), QUERY_PATTERNS["Breast Cancer"])
    all_ok &= check("BreastCancer_records", int(m_bc.sum()),
                    EXPECTED["BreastCancer_records"], results)
    all_ok &= check("BreastCancer_pmids", count_unique_pmids(df, m_bc),
                    EXPECTED["BreastCancer_pmids"], results)

    # --- Homo sapiens (species, word-boundary; canonical single definition) ---
    m_hs = contains_ci(valid_text_series(df["species"]), QUERY_PATTERNS["Homo sapiens"])
    all_ok &= check("HomoSapiens_records", int(m_hs.sum()),
                    EXPECTED["HomoSapiens_records"], results)

    # --- Report ---
    out = pd.DataFrame(results, columns=["Check", "Expected", "Observed", "Status"])
    print("=" * 72)
    print("EVisionary cross-script consistency check (key_B)")
    print("=" * 72)
    print(out.to_string(index=False))
    print("=" * 72)

    n_fail = (out["Status"] == "FAIL").sum()
    if all_ok:
        print("RESULT: ALL CONSISTENT. Counts are identical across the pipeline.")
        sys.exit(0)
    else:
        print(f"RESULT: {n_fail} DRIFT(S) DETECTED. Investigate before freezing outputs.")
        sys.exit(1)


if __name__ == "__main__":
    exit_code = main()         # main returns 0 or 1 instead of sys.exit
    if not hasattr(sys, "ps1") and "ipykernel" not in sys.modules:
        sys.exit(exit_code)

EVisionary cross-script consistency check (key_B)
               Check  Expected  Observed Status
        dataset_rows    258460    258460   PASS
     Protein_records    207623    207623   PASS
       Protein_pmids       569       569   PASS
        mRNA_records     26701     26701   PASS
          mRNA_pmids        26        26   PASS
       miRNA_records     16131     16131   PASS
         miRNA_pmids       120       120   PASS
       Lipid_records      2896      2896   PASS
         Lipid_pmids        52        52   PASS
      Plasma_records       547       547   PASS
        Plasma_pmids       444       444   PASS
BreastCancer_records        12        12   PASS
  BreastCancer_pmids        11        11   PASS
 HomoSapiens_records    200644    200644   PASS
RESULT: ALL CONSISTENT. Counts are identical across the pipeline.


SystemExit: 0

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
